# 05 — Convolutional Neural Networks (CNN)

This notebook trains convolutional networks **from scratch** on the H&M images. Convolutions share weights across positions and preserve the spatial structure that the Dense model discards, so a CNN can be seen as a learned **feature extractor followed by a classifier**.

Starting from a small Conv–ReLU–Pool stack, the model is progressively extended with batch normalisation, dropout, global average pooling and **residual (skip) connections** in the spirit of ResNet. Feature maps and filters are visualised to understand what each layer learns about garments (edges, textures, silhouettes).

> **Status: skeleton.** Only the introduction, objectives and work preview are written. Code and results will be added when the phase is developed.

> **Estado de ejecución verificado · 20/11/2026.** Plan de trabajo: todavía sin implementación ni resultados de entrenamiento. Las ablaciones de redes pendientes no se presentan como realizadas.

## Vía A · CNN desde cero y posible atención

CNN es una arquitectura, mientras que scratch describe cómo inicializamos sus pesos. Aquí se estudia la convolución espacial en imágenes; el notebook 07 reutilizará pesos preentrenados como otra vía. Que la CNN pequeña gane o pierda frente a Dense importa menos que explicar qué aporta su estructura local y el peso compartido.

La pista «CNN + attention layer» puede interpretarse como una extensión: añadir atención sobre canales o posiciones, o transformar posiciones de un mapa de características en tokens. No es un requisito para que una red sea CNN. Una CNN1D sobre word embeddings pertenece a la rama NLP opcional de 11 y no es este experimento visual. Una híbrida CNN + LSTM/GRU también es posible, pero no convierte las unidades recurrentes en convoluciones. Si se añade atención, comparar con la misma CNN sin ella mediante una ablación, sin prometer mejoras. [Fundamento del mecanismo de atención](https://arxiv.org/abs/1706.03762).


## Protocolo común del proyecto

Primero definimos las prendas que queremos clasificar. Después aplicamos los mismos controles de calidad a todas las fotos y conservamos **todas las que los superan**, aunque unas clases tengan más ejemplos. No se fija un número total ni se igualan las clases con cupos.

Las particiones se construyen sobre todos los grupos de producto elegibles. Los productos ya utilizados durante el desarrollo conservan su exposición: un producto de train no se convierte en un supuesto test nuevo. Los grupos nuevos completan aproximadamente 70/15/15 por clase. Se publican los recuentos reales.

**Antes de recortar train por coste, hay que medirlo.** El baseline usa todos los productos elegibles; 04 registra una época piloto y la estimación de tiempo/memoria. Si fuese necesario reducir datos en otro experimento, se justificaría con esa medición, solo en train y conservando validation/test. Una fotografía fuera de una muestra de cálculo no sería una fotografía de mala calidad.

La referencia es `config/experimental_protocol_v1.json`: macro-F1 primaria, accuracy y resultados por clase; pérdida inicial sin pesos, sin aumentos; normalización solo con train a la resolución de entrada. Tres semillas y coste computacional. Los pesos de clase se estudian como ablación: con frecuencias desiguales ya no suponemos que sean casi uno. El test predictivo se reserva hasta congelar las decisiones. Las métricas describen las doce clases elegidas tras los controles, no todo el catálogo.

## Objectives

- Build a baseline CNN (Conv → ReLU → Pool blocks + Dense softmax head) with the shared data pipeline.
- Compare it with the Dense model in accuracy, macro F1, parameters and training time.
- Add batch normalisation, dropout and global average pooling and observe their effect on training stability.
- Implement a small residual block and test whether skip connections help deeper networks train.
- Visualise learned filters and feature maps for example garments of several classes.
- Select the best from-scratch CNN for the final comparison.

## Preview of the work

1. **Setup.** loaders at a CNN resolution (e.g. 128×128 RGB) and training utilities from `src/training/`.
2. **Baseline CNN.** architecture, receptive field, number of parameters.
3. **Training and learning curves.** comparison with the Dense baseline.
4. **Architectural improvements.** BatchNorm, dropout, global average pooling, depth.
5. **Residual blocks.** a mini-ResNet and the link between skip connections and gradient flow.
6. **Interpretation.** filters, feature maps per layer, misclassified examples.
7. **Validation results and conclusions.** best from-scratch CNN.

## Inputs, outputs and course references

| | |
|---|---|
| **Inputs** | Loaders and configuration from notebook 02; Dense results from notebook 04 |
| **Outputs** | Metrics, histories, checkpoints and feature-map figures in `results/` |
| **Course material** | `PRDL_MLLB_CNN.pptx`, `PRDL_MLLB_RestNets_to_NeuralODE.pptx`, `MLLB_CNN_MNIST_in_Keras.ipynb`, `Gemini_MNIST_Pytorch_featuremaps.ipynb` |